In [1]:
import pandas as pd

movies = pd.read_csv("../data/tmdb_5000_movies.csv")
credits = pd.read_csv("../data/tmdb_5000_credits.csv")

print("Movies shape:", movies.shape)
print("Credits shape:", credits.shape)

Movies shape: (4803, 20)
Credits shape: (4803, 4)


In [ ]:
movies.head()

In [ ]:
credits.head()

In [2]:
print("Movies columns:")
print(movies.columns.tolist())

print("\nCredits columns:")
print(credits.columns.tolist())

Movies columns:
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']

Credits columns:
['movie_id', 'title', 'cast', 'crew']


In [3]:
movies = movies.merge(
    credits,
    left_on="id",
    right_on="movie_id"
)

print("New shape:", movies.shape)

New shape: (4803, 24)


In [4]:
movies[["title_x", "genres", "keywords", "cast", "crew"]].head()

,title_x,genres,keywords,cast,crew
0,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,Pirates of the Caribbean: At World's End,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,Spectre,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,The Dark Knight Rises,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,John Carter,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [6]:
import ast

def convert(obj):
    L = []
    for item in ast.literal_eval(obj):
        L.append(item["name"])
    return L


def convert_cast(obj):
    L = []
    for item in ast.literal_eval(obj)[:3]:
        L.append(item["name"])
    return L


movies["genres"] = movies["genres"].apply(convert)
movies["keywords"] = movies["keywords"].apply(convert)
movies["cast"] = movies["cast"].apply(convert_cast)

ValueError: malformed node or string: ['Action', 'Adventure', 'Fantasy', 'Science Fiction']

In [7]:
import ast

def convert(obj):
    if isinstance(obj, list):
        return [item["name"] for item in obj]
    
    if isinstance(obj, str):
        return [item["name"] for item in ast.literal_eval(obj)]
    
    return []


def convert_cast(obj):
    if isinstance(obj, list):
        return obj
    
    if isinstance(obj, str):
        return [item["name"] for item in ast.literal_eval(obj)[:3]]
    
    return []


movies["genres"] = movies["genres"].apply(convert)
movies["keywords"] = movies["keywords"].apply(convert)
movies["cast"] = movies["cast"].apply(convert_cast)

TypeError: string indices must be integers, not 'str'

In [1]:
import pandas as pd

movies = pd.read_csv("../data/tmdb_5000_movies.csv")
credits = pd.read_csv("../data/tmdb_5000_credits.csv")

In [2]:
movies = movies.merge(
    credits,
    left_on="id",
    right_on="movie_id"
)

In [3]:
import ast

def parse_names(value):
    if pd.isna(value):
        return []
    
    data = ast.literal_eval(value)
    return [item["name"] for item in data]


def parse_cast(value):
    if pd.isna(value):
        return []
    
    data = ast.literal_eval(value)
    return [item["name"] for item in data[:3]]


movies["genres"] = movies["genres"].apply(parse_names)
movies["keywords"] = movies["keywords"].apply(parse_names)
movies["cast"] = movies["cast"].apply(parse_cast)

In [4]:
movies[["title_x", "genres", "keywords", "cast", "crew"]].head()

,title_x,genres,keywords,cast,crew
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,Spectre,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,John Carter,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [7]:
def get_director(value):
    if pd.isna(value):
        return []

    # जर आधीच list असेल
    if isinstance(value, list):
        # आधीच director name असेल
        if all(isinstance(item, str) for item in value):
            return value

        # original crew dictionaries असतील
        for item in value:
            if item.get("job") == "Director":
                return [item["name"]]

        return []

    # जर original JSON string असेल
    data = ast.literal_eval(value)

    for item in data:
        if item["job"] == "Director":
            return [item["name"]]

    return []


movies["crew"] = movies["crew"].apply(get_director)

ValueError: The truth value of an empty array is ambiguous. Use `array.size > 0` to check that an array is not empty.

In [8]:
import pandas as pd
import ast

movies = pd.read_csv("../data/tmdb_5000_movies.csv")
credits = pd.read_csv("../data/tmdb_5000_credits.csv")

movies = movies.merge(
    credits,
    left_on="id",
    right_on="movie_id"
)

def parse_names(value):
    if pd.isna(value):
        return []
    data = ast.literal_eval(value)
    return [item["name"] for item in data]

def parse_cast(value):
    if pd.isna(value):
        return []
    data = ast.literal_eval(value)
    return [item["name"] for item in data[:3]]

def get_director(value):
    if pd.isna(value):
        return []
    data = ast.literal_eval(value)
    for item in data:
        if item["job"] == "Director":
            return [item["name"]]
    return []

movies["genres"] = movies["genres"].apply(parse_names)
movies["keywords"] = movies["keywords"].apply(parse_names)
movies["cast"] = movies["cast"].apply(parse_cast)
movies["crew"] = movies["crew"].apply(get_director)

movies[["title_x", "genres", "keywords", "cast", "crew"]].head()

,title_x,genres,keywords,cast,crew
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,Spectre,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan]
4,John Carter,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton]


In [9]:
movies["overview"] = movies["overview"].fillna("")

movies["tags"] = (
    movies["overview"] + " " +
    movies["genres"].apply(lambda x: " ".join(x)) + " " +
    movies["keywords"].apply(lambda x: " ".join(x)) + " " +
    movies["cast"].apply(lambda x: " ".join(x)) + " " +
    movies["crew"].apply(lambda x: " ".join(x))
)

movies[["title_x", "tags"]].head()

,title_x,tags
0,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,Spectre,A cryptic message from Bond’s past sends him o...
3,The Dark Knight Rises,Following the death of District Attorney Harve...
4,John Carter,"John Carter is a war-weary, former military ca..."


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(movies["tags"])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (4803, 28390)


In [11]:
similarity = cosine_similarity(tfidf_matrix)

print("Similarity Matrix Shape:", similarity.shape)

Similarity Matrix Shape: (4803, 4803)


In [12]:
def recommend(movie):
    index = movies[movies["title_x"].str.lower() == movie.lower()].index[0]
    
    distances = similarity[index]
    
    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]
    
    for i in movie_list:
        print(movies.iloc[i[0]]["title_x"])

In [13]:
recommend("Avatar")

Aliens
Alien³
Mission to Mars
Moonraker
Alien


In [14]:
movies = movies.rename(columns={"title_x": "title"})

movies = movies[[
    "movie_id",
    "title",
    "overview",
    "genres",
    "keywords",
    "cast",
    "crew",
    "tags"
]].copy()

movies.head()

,movie_id,title,overview,genres,keywords,cast,crew,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron],"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski],"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes],A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan],Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton],"John Carter is a war-weary, former military ca..."


In [15]:
def recommend(movie):
    index = movies[movies["title"].str.lower() == movie.lower()].index[0]

    distances = similarity[index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    return [movies.iloc[i[0]]["title"] for i in movie_list]

In [16]:
recommend("Avatar")

['Aliens', 'Alien³', 'Mission to Mars', 'Moonraker', 'Alien']